# `tbl-datasets`: training, validation, and held-out test datasets

In [6]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ASSET_NAME = "tbl-datasets"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
if not (PAPER_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from manuscript-training-assets/; {PAPER_DIR} has no microdata_analysis.py")
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/tbl-datasets.*


## Labels and formatting (edit wording here)

In [11]:
ma.setup_analysis_env()
from beta_ols import berke2023 as ds_berke
from beta_ols import covid as ds_covid
from beta_ols import grummon2025 as ds_grummon
from beta_ols import kantorowicz2026 as ds_kantorowicz
from beta_ols import kriner2026 as ds_kriner
from beta_ols import magistro2025 as ds_magistro
from beta_ols import wu2025 as ds_wu

HEADERS = ["Dataset", "Price attributes", "Non-price attributes", "$K$", "$N$", "Tasks", "$d$"]
# Alignment only (':---' left, '---:' right); pipe_table sets the dash counts that give column widths.
COLUMN_RULES = [":---", ":---", ":---", "---:", "---:", "---:", "---:"]
# Each role becomes an italic group row above its datasets (like the model rows in tbl-results-mse).
ROLE_ORDER = ["Training", "Validation", "Test"]

# One row each, in this order: (data module, respondent id column, role, dataset label, citation key, price text, non-price text,
# K = number of response options per choice task).
DATASET_ROWS = [
    (ds_wu, "CaseID", "Training", "Electricity", "wu2025partisan", "Monthly bill ($105 to $125)", "Renewable share (10% to 100%), local renewables", 2),
    (ds_berke, "ResponseId", "Training", "Drones", "berke2023drone", "Delivery fee ($0 to $10)", "Wait time, drone privacy", 2),
    (ds_kriner, "respondent", "Validation", "Flights", "goldfarb2026comfort", "Ticket price ($250 to $325)", "Sustainable aviation fuel, airline, baggage, legroom, seat", 2),
    (ds_kantorowicz, "CASEID", "Test", "Consumer rights", "kantorowicz2026consumers", "Laptop ($500 to $2,000) and laundry ($400 to $1,400) price", "Brand, specs, warranty, privacy, liability, conflict resolution", 2),
    (ds_magistro, "CASEID", "Test", "AI & offshoring", "magistro2026attitudes", "Smartphone price ($300 to $600)", "Customer service, factory, and data science jobs", 5),
    (ds_grummon, "participantid", "Test", "Food labels", "grummon2025nutrition", "None", "Front-of-package warning labels (6 arms)", 3),
    (ds_covid, "respondent", "Test", "Vaccines", "kreps2020factors", "None", "Efficacy, duration, side effects, FDA, origin, endorsements", 3),
]

# How Word shows each citation (informs.csl: "et al." from 3 authors), used only to size the Dataset column.
CITATION_SHOWN = {
    "wu2025partisan": "(Wu 2025)",
    "berke2023drone": "(Berke et al. 2023)",
    "goldfarb2026comfort": "(Goldfarb et al. 2026)",
    "kantorowicz2026consumers": "(Kantorowicz-Reznichenko et al. 2026)",
    "magistro2026attitudes": "(Magistro et al. 2026)",
    "grummon2025nutrition": "(Grummon et al. 2025)",
    "kreps2020factors": "(Kreps et al. 2020)",
}

## Build the table (counts come from the data)

In [12]:
import math
import re

from matplotlib import font_manager
from PIL import ImageFont

# Page and table style of manuscript/quarto-files/ms_template.docx. Word fixes the column widths that Pandoc
# derives from the rule dash counts, so a column narrower than its longest word splits that word across lines.
TEXT_WIDTH_PT = 468.0  # 8.5 in page minus 1 in left and right margins
TABLE_FONT_PT = 10.0  # Compact paragraph style used in table cells
CELL_PADDING_PT = 10.8  # Table style left + right cell margins (108 twips each)
COLUMN_SLACK_PT = 2.0  # per-column allowance for differences between Word and the font metrics measured here
MATH_WIDTH_FACTOR = 1.15  # Word sets $...$ in Cambria Math, which runs wider than Times New Roman
# No-break spaces around ×: Word sets each $...$ as one equation that never wraps.
MATH_SYMBOLS = {r"\beta": "β", r"\rho": "ρ", r"\times": "\u00a0×\u00a0"}


def load_table_font(weight):
    font_properties = font_manager.FontProperties(family="Times New Roman", weight=weight)
    font_path = font_manager.findfont(font_properties, fallback_to_default=False)
    return ImageFont.truetype(font_path, 1000)


REGULAR_FONT = load_table_font("normal")
BOLD_FONT = load_table_font("bold")


def shown_math(match):
    latex = match.group(1)
    text = latex
    for command, symbol in MATH_SYMBOLS.items():
        text = text.replace(command, symbol)
    text = text.replace(r"\mathrm", "")
    for mark in ("{", "}", "_", "^"):
        text = text.replace(mark, "")
    if "\\" in text:
        raise ValueError(f"No display text for a LaTeX command in {latex!r}; add it to MATH_SYMBOLS")
    return text


def shown_citation(match):
    citation_key = match.group(1)
    if citation_key not in CITATION_SHOWN:
        raise KeyError(f"Add {citation_key!r} to CITATION_SHOWN in the labels cell")
    return CITATION_SHOWN[citation_key]


def shown_cell(cell):
    """Text Word displays for a markdown cell, with the font and width factor to measure it."""
    is_bold = cell.startswith("**") and cell.endswith("**")
    is_italic = not is_bold and cell.startswith("*") and cell.endswith("*") and len(cell) > 1
    text = cell
    if is_bold:
        text = text[2:-2]
    if is_italic:
        text = text[1:-1]
    text = re.sub(r"\[@([^\]]+)\]", shown_citation, text)
    has_math = re.search(r"(?<!\\)\$", text) is not None
    text = re.sub(r"(?<!\\)\$(.+?)(?<!\\)\$", shown_math, text)
    text = text.replace(r"\$", "$")
    text = text.replace(r"\|", "|")
    font = BOLD_FONT if is_bold else REGULAR_FONT
    factor = MATH_WIDTH_FACTOR if has_math else 1.0
    return text, font, factor


def text_width_pt(text, font, factor):
    return font.getlength(text) / 1000.0 * TABLE_FONT_PT * factor


def unbreakable_pieces(text):
    """Pieces Word keeps on one line: words, with compound words (letter-hyphen-letter) split after the hyphen."""
    pieces = []
    for word in re.split(r"[ \t\n]+", text):
        pieces.extend(re.split(r"(?<=[A-Za-z]-)(?=[A-Za-z])", word))
    return pieces


def column_widths_pt(headers, rows):
    """Each column gets at least its widest unbreakable piece; leftover page width goes to columns that wrap."""
    n_columns = len(headers)
    min_pt = [0.0] * n_columns
    natural_pt = [0.0] * n_columns
    for cells in [headers, *rows]:
        if len(cells) != n_columns:
            raise ValueError(f"Row has {len(cells)} cells, expected {n_columns}: {cells}")
        for column_index, cell in enumerate(cells):
            text, font, factor = shown_cell(cell)
            for piece in unbreakable_pieces(text):
                piece_pt = text_width_pt(piece, font, factor)
                min_pt[column_index] = max(min_pt[column_index], piece_pt)
            cell_pt = text_width_pt(text, font, factor)
            natural_pt[column_index] = max(natural_pt[column_index], cell_pt)
    extra_pt = CELL_PADDING_PT + COLUMN_SLACK_PT
    min_pt = [width + extra_pt for width in min_pt]
    natural_pt = [width + extra_pt for width in natural_pt]
    if sum(min_pt) > TEXT_WIDTH_PT:
        raise ValueError(
            f"Longest words need {sum(min_pt):.0f}pt but the page is {TEXT_WIDTH_PT:.0f}pt wide; "
            f"shorten labels or drop columns. Per-column minimum: {[round(width) for width in min_pt]}"
        )
    if sum(natural_pt) <= TEXT_WIDTH_PT:
        return natural_pt
    spare_pt = TEXT_WIDTH_PT - sum(min_pt)
    stretch_pt = [natural - minimum for natural, minimum in zip(natural_pt, min_pt)]
    total_stretch_pt = sum(stretch_pt)
    widths_pt = []
    for minimum, stretch in zip(min_pt, stretch_pt):
        widths_pt.append(minimum + spare_pt * stretch / total_stretch_pt)
    return widths_pt


def escape_markdown(text):
    """Escape $ and | so Pandoc reads them as text inside a pipe table."""
    escaped = text.replace("$", r"\$")
    escaped = escaped.replace("|", r"\|")
    return escaped


def pipe_table(headers, rows, column_rules):
    """Pandoc pipe table. column_rules set alignment (':---' left, '---:' right); dash counts come from column_widths_pt."""
    if len(column_rules) != len(headers):
        raise ValueError(f"{len(column_rules)} column rules for {len(headers)} columns")
    widths_pt = column_widths_pt(headers, rows)
    rule_cells = []
    for column_rule, width_pt in zip(column_rules, widths_pt):
        if re.fullmatch(r":?-+:?", column_rule) is None:
            raise ValueError(f"Bad column rule {column_rule!r}")
        dashes = "-" * round(width_pt)
        left = ":" if column_rule.startswith(":") else ""
        right = ":" if column_rule.endswith(":") else ""
        rule_cells.append(left + dashes + right)
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("|" + "|".join(rule_cells) + "|")
    for row in rows:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines) + "\n"

In [13]:
rows = []
roles_in_table = []
for module, respondent_column, role, dataset_label, citation_key, price_text, nonprice_text, n_options in DATASET_ROWS:
    if role not in ROLE_ORDER:
        raise ValueError(f"Role {role!r} is not in ROLE_ORDER")
    if role not in roles_in_table:
        if roles_in_table and ROLE_ORDER.index(role) < ROLE_ORDER.index(roles_in_table[-1]):
            raise ValueError(f"DATASET_ROWS must follow ROLE_ORDER; {role!r} comes after {roles_in_table[-1]!r}")
        roles_in_table.append(role)
        group_row = [f"*{role}*"] + [""] * (len(HEADERS) - 1)
        rows.append(group_row)
    choice_df = module.load_choice_df()
    n_respondents = int(choice_df[respondent_column].nunique())
    n_tasks = int(len(choice_df))
    specs = module.regression_specs(module.regression_rows(choice_df))
    n_slopes = 0
    for spec in specs.values():
        n_slopes += len(spec.target_labels)
    rows.append([
        f"{dataset_label} [@{citation_key}]",
        escape_markdown(price_text),
        escape_markdown(nonprice_text),
        f"{n_options}",
        f"{n_respondents:,}",
        f"{n_tasks:,}",
        f"{n_slopes}",
    ])

table_md = pipe_table(HEADERS, rows, COLUMN_RULES)
display(pd.DataFrame(rows, columns=HEADERS))

,Dataset,Price attributes,Non-price attributes,$K$,$N$,Tasks,$d$
0,*Training*,,,,,,
1,Electricity [@wu2025partisan],Monthly bill (\$105 to \$125),"Renewable share (10% to 100%), local renewables",2,"1,861","18,591",8
2,Drones [@berke2023drone],Delivery fee (\$0 to \$10),"Wait time, drone privacy",2,"3,625","36,297",28
3,*Validation*,,,,,,
4,Flights [@goldfarb2026comfort],Ticket price (\$250 to \$325),"Sustainable aviation fuel, airline, baggage, l...",2,"1,000","5,000",12
5,*Test*,,,,,,
6,Consumer rights [@kantorowicz2026consumers],"Laptop (\$500 to \$2,000) and laundry (\$400 t...","Brand, specs, warranty, privacy, liability, co...",2,"2,242","26,823",38
7,AI & offshoring [@magistro2026attitudes],Smartphone price (\$300 to \$600),"Customer service, factory, and data science jobs",5,"5,525","21,005",32
8,Food labels [@grummon2025nutrition],None,Front-of-package warning labels (6 arms),3,"5,633","33,798",5
9,Vaccines [@kreps2020factors],None,"Efficacy, duration, side effects, FDA, origin,...",3,"1,971","9,855",11


## Save `tbl-datasets.md` (included by the manuscript)

In [14]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

| Dataset | Price attributes | Non-price attributes | $K$ | $N$ | Tasks | $d$ |
|:---------------------------------------------------------------------------------------------------------------------|:-----------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------|---------------------:|-----------------------------------:|----------------------------------------:|-----------------------:|
| *Training* |  |  |  |  |  |  |
| Electricity [@wu2025partisan] | Monthly bill (\$105 to \$125) | Renewable share (10% to 100%), local renewables | 2 | 1,861 | 18,591 | 8 |
| Drones [@berke2023drone] | Delivery fee (\$0 to \$10) | Wait time, drone privacy | 2 | 3,625 | 36,297 | 28 |
| *Validation* |  |  |  |  |  |  |
| Flights [@goldfarb2026comfort] | Ticket price (\$250 to \$325) | Sustainable aviation fuel, airline, baggage, legro